# 00 - Shared Drive data

The big Zenodo files already live in the shared Drive folder `Cadenza Challenge`, so nobody re-downloads them from Zenodo.

**One-time (per person):** open the shared folder in Drive -> right-click -> *Organize* -> *Add shortcut* -> **My Drive** (keep the name `Cadenza Challenge`).

These files must sit directly in that folder (not in a subfolder):
`metadata.zip`, `EnsembleSet_Mix_1.zip`, `CadenzaWoodwind.zip`, `cadenza_cad2_task2_eval.v1_1.tar.gz`

- **Part A** - copy the zips from Drive to Colab's local disk (fast), check md5, and build the dataset.
- **Part B** (optional) - copy and extract the eval set used by notebook 02.

This notebook clones and installs Clarity itself if `/content/clarity` is missing. Needs ~30 GB free on the Colab disk.
No Google Drive app is needed on your laptop - Colab mounts Drive in the cloud.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
SHARED = Path('/content/drive/MyDrive/Cadenza Challenge')   # the shortcut name must match
assert SHARED.exists(), 'Add a shortcut to the shared folder in My Drive first (see top of notebook)'
print('shared folder ok:', SHARED)
print(sorted(p.name for p in SHARED.iterdir()))

Mounted at /content/drive
shared folder ok: /content/drive/MyDrive/Cadenza Challenge
['CadenzaWoodwind.zip', 'EnsembleSet_Mix_1.zip', 'cadenza_cad2_task2_eval.v1_1.tar.gz', 'metadata.zip']


In [2]:
import hashlib, shutil

FILES = {   # name -> md5 (same values as notebooks 02 and 03)
    'metadata.zip': '7ff49002ae62023cd547ff057cbba461',
    'EnsembleSet_Mix_1.zip': '5bdcd0154da69e7516aef2d3062a47f0',
    'CadenzaWoodwind.zip': 'cdc08be8482662c5023b5d4060d1b064',
    'cadenza_cad2_task2_eval.v1_1.tar.gz': '54268aa8dc08035cd012483bba701d09',
}

def md5sum(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

missing = [n for n in FILES if not (SHARED / n).exists()]
assert not missing, f'Not found directly in {SHARED}: {missing} (are they in a subfolder?)'
print('all files found')

all files found


## Part A - copy from Drive and build the dataset (everyone)
Takes a few minutes. Copies skip files that are already local and verified.

In [3]:
dl = Path('/content/zenodo_download'); dl.mkdir(exist_ok=True)
for name, want in FILES.items():
    if name.endswith('.tar.gz'):
        continue                      # eval set is handled in Part B
    dst = dl / name
    if not (dst.exists() and md5sum(dst) == want):
        print('copying', name)
        shutil.copy(SHARED / name, dst)
    assert md5sum(dst) == want, f'md5 mismatch for {name} - re-upload it to the shared folder'
    print('OK', name)

copying metadata.zip
OK metadata.zip
copying EnsembleSet_Mix_1.zip


KeyboardInterrupt: 

In [ ]:
import subprocess
if not Path('/content/clarity').exists():   # same as 01_setup.ipynb, so this notebook works on its own
    subprocess.run(['git', 'clone', 'https://github.com/claritychallenge/clarity.git', '/content/clarity'], check=True)
    subprocess.run(['pip', 'install', '-q', '-e', '/content/clarity'], check=True)
print('clarity ready:', Path('/content/clarity').exists())


In [ ]:
import os, subprocess
root = Path('/content/cadenza_data/cad2/task2')
clarity_dir = Path('/content/clarity')
assert clarity_dir.exists(), 'Run the Clarity clone cell above first'
script = clarity_dir / 'recipes/cad2/task2/process_dataset/process_zenodo_download.py' # Section 1.1 of third_party/clarity/recipes/cad2/task2/README.md
assert script.exists(), f'missing {script}'
# Colab's torchtune has its own `recipes` package; PYTHONPATH makes Clarity's win.
env = {**os.environ, 'PYTHONPATH': str(clarity_dir)}
r = subprocess.run(['python', str(script), f'path.zenodo_download_path={dl}', f'path.root={root}'], # Section 1.1 of third_party/clarity/recipes/cad2/task2/README.md
                   env=env, capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
r.check_returncode()   # fail loudly here instead of at the check below

In [ ]:
meta = root / 'metadata'
print('root exists:', root.exists(), '| metadata exists:', meta.exists())
if meta.exists(): print(sorted(p.name for p in meta.iterdir()))
for name in ['scenes.valid.json', 'scene_listeners.valid.json', 'music.valid.json', 'listeners.valid.json']:
    assert (meta / name).exists(), f'missing {name}'
print('dataset ready:', root)

## Part B - eval set (optional, used by notebook 02)
Copied from the shared Drive folder and extracted on Colab's local disk.

In [ ]:
EV = 'cadenza_cad2_task2_eval.v1_1.tar.gz'
dst = Path('/content') / EV
if not (dst.exists() and md5sum(dst) == FILES[EV]):
    shutil.copy(SHARED / EV, dst)
assert md5sum(dst) == FILES[EV], 'eval md5 mismatch - re-upload it to the shared folder'
!cd /content && tar -xzf {EV} && ls cadenza_data/cad2/task2